## 1. Cấu hình

In [4]:
class CFG:
    EMBEDDING_DIM = 16
    COMPRESSION_RATIO = 0.25
    NUM_HASH_FUNCTIONS = 4
    EPOCHS = 1
    BATCH_SIZE = 4096
    LEARNING_RATE = 1e-3
    USE_AMP = True
    SAMPLE_FRAC = 1.0
    RESUME = True
    USE_BATCHNORM = False
    NUM_DENSE = 13
    NUM_CAT = 26
    DATA_DIR = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2"
    OUTPUT_DIR = "/kaggle/working"
    MODELS = ["deepfm", "dcn", "dlrm"]
    DEEPFM = {"mlp_dims": [400, 400, 400], "dropout": 0.1}
    DCN = {"num_cross_layers": 3, "mlp_dims": [400, 400], "dropout": 0.1}
    DLRM = {"bottom_mlp_dims": [256, 128], "top_mlp_dims": [256, 128], "dropout": 0.1}

## 2. Import thư viện

In [5]:
import os
import json
import time
import hashlib
import csv

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
import torch.nn as nn

from sklearn.utils import murmurhash3_32
from sklearn.metrics import roc_auc_score, log_loss

## 3. Bloom embedding

In [6]:
# 24 seed cố định -> mỗi seed là 1 hàm hash độc lập.
SEEDS = [
    179424941, 179425457, 179425907, 179426369,
    179424977, 179425517, 179425943, 179426407,
    179424989, 179425529, 179425993, 179426447,
    179425003, 179425537, 179426003, 179426453,
    179425019, 179425559, 179426029, 179426491,
    179425027, 179425579, 179426081, 179426549,
]


class BloomEmbedding(nn.Module):
    """Bảng embedding nén: hash mỗi id qua k hàm hash vào bảng nhỏ rồi cộng vector.

    Hàng 0 của bảng nén được dành riêng cho padding/OOV (vector 0); các id khác
    hash vào [1, compressed-1]. Index 0 đầu vào luôn trả về vector 0.
    """

    def __init__(self, num_embeddings, embedding_dim, compression_ratio=0.25,
                 num_hash_functions=4, padding_idx=0):
        super().__init__()
        if num_hash_functions > len(SEEDS):
            raise ValueError(f"Tối đa {len(SEEDS)} hàm hash ({num_hash_functions} yêu cầu)")
        self.num_embeddings = int(num_embeddings)
        self.num_hash_functions = num_hash_functions
        self.padding_idx = padding_idx
        # Tối thiểu 2 hàng: hàng 0 (padding) + ít nhất 1 hàng dữ liệu.
        self.compressed = max(2, int(compression_ratio * self.num_embeddings))

        self.embeddings = nn.Embedding(self.compressed, embedding_dim, padding_idx=0)
        nn.init.normal_(self.embeddings.weight, mean=0.0, std=1.0 / embedding_dim)
        with torch.no_grad():
            self.embeddings.weight[0].fill_(0.0)

        self.register_buffer("_hashes", self._build_hashes())

    def _build_hashes(self):
        idx = np.arange(self.num_embeddings, dtype=np.int32)
        cols = []
        for seed in SEEDS[:self.num_hash_functions]:
            h = murmurhash3_32(idx, seed=seed, positive=True) % (self.compressed - 1) + 1
            cols.append(h)
        hashes = np.stack(cols, axis=1).astype(np.int64)  # (num_embeddings, k)
        hashes[self.padding_idx] = 0                       # padding -> hàng 0
        return torch.from_numpy(hashes)

    def forward(self, indices):
        # indices: (batch,) long -> (batch, dim)
        hashed = self._hashes[indices]          # (batch, k)
        emb = self.embeddings(hashed)           # (batch, k, dim)
        return emb.sum(dim=1)                   # (batch, dim)


class EmbeddingTable(nn.Module):
    """Gom nhiều BloomEmbedding, mỗi field 1 bảng, cùng embedding_dim."""

    def __init__(self, vocab_sizes, embedding_dim, compression_ratio, num_hash_functions):
        super().__init__()
        self.num_fields = len(vocab_sizes)
        self.embedding_dim = embedding_dim
        self.embs = nn.ModuleList([
            BloomEmbedding(v, embedding_dim, compression_ratio, num_hash_functions)
            for v in vocab_sizes
        ])

    def forward(self, cat):
        # cat: (batch, num_fields) long -> (batch, num_fields, dim)
        out = [self.embs[i](cat[:, i]) for i in range(self.num_fields)]
        return torch.stack(out, dim=1)

## 4. Định nghĩa model

DeepFM, DCN, DLRM - dùng chung lớp Bloom embedding.

In [7]:
class MLP(nn.Module):
    def __init__(self, in_dim, dims, dropout=0.0, use_bn=False):
        super().__init__()
        layers = []
        for d in dims:
            layers.append(nn.Linear(in_dim, d))
            if use_bn:
                layers.append(nn.BatchNorm1d(d))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = d
        self.net = nn.Sequential(*layers)
        self.out_dim = dims[-1] if dims else in_dim

    def forward(self, x):
        return self.net(x)

In [8]:
class DeepFM(nn.Module):
    def __init__(self, vocab_sizes, dense_dim, embedding_dim, compression_ratio,
                 num_hash_functions, mlp_dims, dropout=0.0, use_bn=False):
        super().__init__()
        num_fields = len(vocab_sizes)
        self.embed = EmbeddingTable(vocab_sizes, embedding_dim,
                                    compression_ratio, num_hash_functions)
        # Bậc 1: mỗi field 1 vô hướng (bloom embedding dim=1) + dense linear.
        self.cat_linear = EmbeddingTable(vocab_sizes, 1, compression_ratio, num_hash_functions)
        self.dense_linear = nn.Linear(dense_dim, 1)
        self.bias = nn.Parameter(torch.zeros(1))
        deep_in = num_fields * embedding_dim + dense_dim
        self.mlp = MLP(deep_in, mlp_dims, dropout, use_bn)
        self.deep_head = nn.Linear(self.mlp.out_dim, 1)

    def forward(self, cat, dense):
        emb = self.embed(cat)                                   # (B, F, d)
        # FM bậc 2: 0.5 * sum_d((sum emb)^2 - sum(emb^2))
        sum_sq = emb.sum(dim=1).pow(2)                          # (B, d)
        sq_sum = emb.pow(2).sum(dim=1)                          # (B, d)
        fm2 = 0.5 * (sum_sq - sq_sum).sum(dim=1, keepdim=True)  # (B, 1)
        # FM bậc 1
        lin = self.cat_linear(cat).sum(dim=1) + self.dense_linear(dense)  # (B, 1)
        # Deep
        deep = self.deep_head(self.mlp(torch.cat([emb.flatten(1), dense], dim=1)))  # (B, 1)
        return (self.bias + lin + fm2 + deep).squeeze(1)        # (B,)

In [9]:
class CrossNetwork(nn.Module):
    def __init__(self, in_dim, num_layers):
        super().__init__()
        self.w = nn.ModuleList([nn.Linear(in_dim, 1, bias=False) for _ in range(num_layers)])
        self.b = nn.ParameterList([nn.Parameter(torch.zeros(in_dim)) for _ in range(num_layers)])

    def forward(self, x0):
        x = x0
        for i in range(len(self.w)):
            x = x0 * self.w[i](x) + self.b[i] + x   # x_{l+1} = x0 * (w.x_l) + b + x_l
        return x


class DCN(nn.Module):
    def __init__(self, vocab_sizes, dense_dim, embedding_dim, compression_ratio,
                 num_hash_functions, num_cross_layers, mlp_dims, dropout=0.0, use_bn=False):
        super().__init__()
        num_fields = len(vocab_sizes)
        self.embed = EmbeddingTable(vocab_sizes, embedding_dim,
                                    compression_ratio, num_hash_functions)
        in_dim = num_fields * embedding_dim + dense_dim
        self.cross = CrossNetwork(in_dim, num_cross_layers)
        self.mlp = MLP(in_dim, mlp_dims, dropout, use_bn)
        self.head = nn.Linear(in_dim + self.mlp.out_dim, 1)

    def forward(self, cat, dense):
        emb = self.embed(cat).flatten(1)            # (B, F*d)
        x = torch.cat([emb, dense], dim=1)          # (B, in_dim)
        c = self.cross(x)
        d = self.mlp(x)
        return self.head(torch.cat([c, d], dim=1)).squeeze(1)

In [10]:
class DLRM(nn.Module):
    def __init__(self, vocab_sizes, dense_dim, embedding_dim, compression_ratio,
                 num_hash_functions, bottom_mlp_dims, top_mlp_dims, dropout=0.0, use_bn=False):
        super().__init__()
        num_fields = len(vocab_sizes)
        self.embed = EmbeddingTable(vocab_sizes, embedding_dim,
                                    compression_ratio, num_hash_functions)
        # Bottom MLP ép dense về đúng embedding_dim để dot-product được.
        self.bottom = MLP(dense_dim, list(bottom_mlp_dims) + [embedding_dim], dropout, use_bn)
        num_feat = num_fields + 1                    # 26 cat emb + 1 dense emb
        num_inter = num_feat * (num_feat - 1) // 2   # tam giác trên, bỏ đường chéo
        top_in = embedding_dim + num_inter
        self.top = MLP(top_in, top_mlp_dims, dropout, use_bn)
        self.head = nn.Linear(self.top.out_dim, 1)
        self.register_buffer("_triu", torch.triu_indices(num_feat, num_feat, offset=1))

    def forward(self, cat, dense):
        emb = self.embed(cat)                        # (B, F, d)
        de = self.bottom(dense).unsqueeze(1)         # (B, 1, d)
        feats = torch.cat([de, emb], dim=1)          # (B, F+1, d)
        inter = torch.bmm(feats, feats.transpose(1, 2))          # (B, F+1, F+1)
        inter = inter[:, self._triu[0], self._triu[1]]           # (B, num_inter)
        x = torch.cat([de.squeeze(1), inter], dim=1)             # (B, d + num_inter)
        return self.head(self.top(x)).squeeze(1)

## 5. Tiền xử lý dữ liệu

Encode categorical, chuẩn hoá dense, cache lại để chạy lại nhanh.

In [11]:
DENSE_COLS = [f"I{i}" for i in range(1, 14)]
CAT_COLS = [f"C{i}" for i in range(1, 27)]
SPLITS = ["train", "test"]


def build_category_mapping(series):
    """Trả về dict {value: code} với code bắt đầu từ 1 (0 dành cho OOV/padding)."""
    uniques = pd.unique(series.dropna())
    return {v: i + 1 for i, v in enumerate(uniques)}


def apply_category_mapping(series, mapping):
    """Map series -> np.int32; giá trị không có trong mapping -> 0."""
    return series.map(mapping).fillna(0).astype(np.int32).to_numpy()


def standardize(arr, mean, std):
    """Chuẩn hoá (arr - mean) / std, tránh chia 0."""
    safe_std = np.where(std < 1e-8, 1.0, std)
    return ((arr - mean) / safe_std).astype(np.float32)


def _read_column(path, col):
    return pq.read_table(path, columns=[col]).column(col).to_pandas()


def preprocess(config):
    """Sinh cache nếu chưa có. Trả về vocab_sizes (list 26 phần tử)."""
    out = config.OUTPUT_DIR
    os.makedirs(out, exist_ok=True)
    done_marker = os.path.join(out, "_PREPROCESS_DONE")
    vocab_path = os.path.join(out, "vocab_sizes.json")

    if os.path.exists(done_marker) and os.path.exists(vocab_path):
        with open(vocab_path) as f:
            return json.load(f)

    paths = {s: os.path.join(config.DATA_DIR, f"{s}.parquet") for s in SPLITS}

    # 1) Label
    for s in SPLITS:
        label = _read_column(paths[s], "label").astype(np.int8).to_numpy()
        np.save(os.path.join(out, f"label_{s}.npy"), label)
    n_rows = {s: np.load(os.path.join(out, f"label_{s}.npy")).shape[0] for s in SPLITS}

    # 2) Categorical: xử lý từng field để giới hạn RAM đỉnh
    vocab_sizes = []
    cat_arrays = {s: np.zeros((n_rows[s], len(CAT_COLS)), dtype=np.int32) for s in SPLITS}
    for j, col in enumerate(CAT_COLS):
        train_col = _read_column(paths["train"], col)
        mapping = build_category_mapping(train_col)
        vocab_sizes.append(len(mapping) + 1)  # +1 cho index 0
        cat_arrays["train"][:, j] = apply_category_mapping(train_col, mapping)
        del train_col
        for s in ("test",):
            cat_arrays[s][:, j] = apply_category_mapping(_read_column(paths[s], col), mapping)
    for s in SPLITS:
        np.save(os.path.join(out, f"cat_{s}.npy"), cat_arrays[s])
    del cat_arrays

    # 3) Dense: đọc 13 cột, fillna(0), tính mean/std từ train, standardize
    def load_dense(path):
        df = pq.read_table(path, columns=DENSE_COLS).to_pandas()
        return df.fillna(0.0).to_numpy().astype(np.float32)

    dense_train = load_dense(paths["train"])
    mean = dense_train.mean(axis=0)
    std = dense_train.std(axis=0)
    np.savez(os.path.join(out, "dense_scaler.npz"), mean=mean, std=std)
    np.save(os.path.join(out, "dense_train.npy"), standardize(dense_train, mean, std))
    del dense_train
    for s in ("test",):
        np.save(os.path.join(out, f"dense_{s}.npy"), standardize(load_dense(paths[s]), mean, std))

    with open(vocab_path, "w") as f:
        json.dump(vocab_sizes, f)
    open(done_marker, "w").close()
    return vocab_sizes

## 6. Dataset

Nạp dữ liệu vào RAM, chia batch.

In [12]:
class CriteoArrays:
    """Giữ cat/dense/label dạng tensor CPU; cat dạng int32, chuyển long khi tạo batch."""

    def __init__(self, cat, dense, label):
        self.cat = torch.from_numpy(cat)                       # (N, num_cat) int32
        self.dense = torch.from_numpy(dense)                   # (N, num_dense) float32
        self.label = torch.from_numpy(label).float()           # (N,) float32
        self.n = self.label.shape[0]


def load_arrays(output_dir, split, sample_frac=1.0):
    """Nạp cat/dense/label của 1 split. sample_frac < 1.0 -> lấy prefix N*frac dòng."""
    cat = np.load(os.path.join(output_dir, f"cat_{split}.npy"))
    dense = np.load(os.path.join(output_dir, f"dense_{split}.npy"))
    label = np.load(os.path.join(output_dir, f"label_{split}.npy"))
    if sample_frac < 1.0:
        n = int(len(label) * sample_frac)
        cat, dense, label = cat[:n], dense[:n], label[:n]
    return CriteoArrays(cat, dense, label)


def iterate_batches(arrays, batch_size, shuffle, device, generator=None):
    """Sinh (cat_long, dense, label) theo batch. Index slice trên tensor in-RAM."""
    if shuffle:
        order = torch.randperm(arrays.n, generator=generator)
    else:
        order = torch.arange(arrays.n)
    for start in range(0, arrays.n, batch_size):
        b = order[start:start + batch_size]
        cat = arrays.cat[b].long().to(device, non_blocking=True)
        dense = arrays.dense[b].to(device, non_blocking=True)
        y = arrays.label[b].to(device, non_blocking=True)
        yield cat, dense, y

## 7. Huấn luyện & đánh giá

In [13]:
def _measure_flops(model, arrays, device):
    """FLOPs cho 1 mẫu / 1 forward (đếm nn.Linear; 1 MAC = 2 FLOPs)."""
    model.eval()
    total = {"f": 0}
    hooks = []

    def _hook(m, inp, out):
        total["f"] += 2 * m.in_features * m.out_features

    for m in model.modules():
        if isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(_hook))
    try:
        cat = arrays.cat[:1].long().to(device)
        dense = arrays.dense[:1].to(device)
        with torch.no_grad():
            model(cat, dense)
    finally:
        for h in hooks:
            h.remove()
    return total["f"]


def _measure_infer_latency(model, arrays, config, device, n_batches=20):
    """Độ trễ inference trung bình (ms / batch) trên test."""
    model.eval()
    times = []
    n = 0
    with torch.no_grad():
        for cat, dense, y in iterate_batches(arrays, config.BATCH_SIZE, False, device):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            model(cat, dense)
            if device.type == "cuda":
                torch.cuda.synchronize()
            if n > 0:  # bỏ batch đầu (warmup)
                times.append((time.time() - t0) * 1000.0)
            n += 1
            if n > n_batches:
                break
    return sum(times) / len(times) if times else float("nan")


def build_model(name, vocab_sizes, config):
    """Factory: tạo model theo tên, lấy hyperparam từ config."""
    dense_dim = config.NUM_DENSE
    common = dict(vocab_sizes=vocab_sizes, dense_dim=dense_dim,
                  embedding_dim=config.EMBEDDING_DIM,
                  compression_ratio=config.COMPRESSION_RATIO,
                  num_hash_functions=config.NUM_HASH_FUNCTIONS,
                  use_bn=config.USE_BATCHNORM)
    if name == "deepfm":
        return DeepFM(**common, mlp_dims=config.DEEPFM["mlp_dims"],
                      dropout=config.DEEPFM["dropout"])
    if name == "dcn":
        return DCN(**common, num_cross_layers=config.DCN["num_cross_layers"],
                   mlp_dims=config.DCN["mlp_dims"], dropout=config.DCN["dropout"])
    if name == "dlrm":
        return DLRM(**common, bottom_mlp_dims=config.DLRM["bottom_mlp_dims"],
                    top_mlp_dims=config.DLRM["top_mlp_dims"], dropout=config.DLRM["dropout"])
    raise ValueError(f"Model không hỗ trợ: {name}")


def config_hash(name, vocab_sizes, config):
    """Băm các tham số ảnh hưởng kết quả -> vô hiệu checkpoint cũ khi đổi config."""
    payload = {
        "name": name,
        "vocab": list(vocab_sizes),
        "dim": config.EMBEDDING_DIM,
        "cr": config.COMPRESSION_RATIO,
        "k": config.NUM_HASH_FUNCTIONS,
        "lr": config.LEARNING_RATE,
        "bs": config.BATCH_SIZE,
        "epochs": config.EPOCHS,
    }
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()


@torch.no_grad()
def evaluate(model, arrays, config, device):
    """Trả về (auc, logloss) trên toàn bộ arrays."""
    model.eval()
    preds, labels = [], []
    for cat, dense, y in iterate_batches(arrays, config.BATCH_SIZE, False, device):
        logit = model(cat, dense)
        preds.append(torch.sigmoid(logit).float().cpu().numpy())
        labels.append(y.cpu().numpy())
    p = np.concatenate(preds)
    t = np.concatenate(labels)
    return roc_auc_score(t, p), log_loss(t, p, labels=[0, 1])


def _load_ckpt(path, expected_hash):
    if not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location="cpu")
    if ckpt.get("config_hash") != expected_hash:
        return None     # config đổi -> bỏ checkpoint cũ
    return ckpt


def train_model(name, train_arr, test_arr, vocab_sizes, config, device, output_dir):
    """Train 1 model đến đủ EPOCHS (resume nếu có ckpt hợp lệ). Trả về dict metrics."""
    model = build_model(name, vocab_sizes, config).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    emb_params = sum(p.numel() for nm, p in model.named_parameters()
                     if nm.startswith("embed") or nm.startswith("cat_linear"))
    flops = _measure_flops(model, train_arr, device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    try:
        import psutil
        _proc = psutil.Process()
    except Exception:
        _proc = None
    peak_ram = _proc.memory_info().rss if _proc is not None else 0
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    scaler = torch.amp.GradScaler('cuda', enabled=config.USE_AMP and device.type == "cuda")
    criterion = nn.BCEWithLogitsLoss()
    chash = config_hash(name, vocab_sizes, config)
    ckpt_path = os.path.join(output_dir, f"ckpt_{name}.pt")

    start_epoch = 0
    epoch_times = []
    if config.RESUME:
        ckpt = _load_ckpt(ckpt_path, chash)
        if ckpt is not None:
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            scaler.load_state_dict(ckpt["scaler_state"])
            start_epoch = ckpt["epoch"]
            epoch_times = ckpt.get("epoch_times", [])

    for epoch in range(start_epoch, config.EPOCHS):
        model.train()
        t0 = time.time()
        for cat, dense, y in iterate_batches(train_arr, config.BATCH_SIZE, True, device):
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=config.USE_AMP and device.type == "cuda"):
                loss = criterion(model(cat, dense), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        epoch_times.append(time.time() - t0)
        if _proc is not None:
            peak_ram = max(peak_ram, _proc.memory_info().rss)
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scaler_state": scaler.state_dict(),
            "epoch": epoch + 1,
            "epoch_times": epoch_times,
            "config_hash": chash,
        }, ckpt_path)

    auc, logloss = evaluate(model, test_arr, config, device)
    h_per_epoch = (sum(epoch_times) / len(epoch_times) / 3600.0) if epoch_times else 0.0

    # --- Tài nguyên & độ trễ ---
    vram_gb = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else float("nan")
    if _proc is not None:
        peak_ram = max(peak_ram, _proc.memory_info().rss)
    ram_gb = peak_ram / 1e9 if _proc is not None else float("nan")
    infer_ms = _measure_infer_latency(model, test_arr, config, device)

    return {"model": name, "auc": auc, "logloss": logloss,
            "h_per_epoch": h_per_epoch, "epochs": config.EPOCHS,
            "emb_params": emb_params, "n_params": n_params, "flops": flops,
            "ram_gb": ram_gb, "vram_gb": vram_gb, "infer_ms": infer_ms}

## 8. Chạy thử nghiệm

In [14]:
RESULT_FIELDS = ["model", "auc", "logloss", "emb_params", "n_params",
                 "flops", "ram_gb", "vram_gb", "h_per_epoch", "infer_ms", "epochs"]


def _done_models(results_csv):
    if not os.path.exists(results_csv):
        return set()
    with open(results_csv, newline="") as f:
        return {row["model"] for row in csv.DictReader(f)}


def _append_result(results_csv, res):
    exists = os.path.exists(results_csv)
    with open(results_csv, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow({k: res[k] for k in RESULT_FIELDS})


def _write_markdown(results_csv, results_md):
    with open(results_csv, newline="") as f:
        rows = list(csv.DictReader(f))
    lines = ["# Kết quả Bloom Embedding CTR", "",
             "| Model | AUC | LogLoss | Emb Params | Tổng Params | FLOPs | RAM (GB) | VRAM (GB) | Train (h/ep) | Infer (ms/bs) | Epochs |",
             "|---|---|---|---|---|---|---|---|---|---|---|"]
    for r in rows:
        lines.append(f"| {r['model']} | {float(r['auc']):.5f} | "
                     f"{float(r['logloss']):.5f} | "
                     f"{float(r['emb_params'])/1e9:.4f}B | {float(r['n_params'])/1e9:.4f}B | "
                     f"{int(float(r['flops'])):,} | {float(r['ram_gb']):.2f} | "
                     f"{float(r['vram_gb']):.2f} | {float(r['h_per_epoch']):.4f} | "
                     f"{float(r['infer_ms']):.2f} | {r['epochs']} |")
    with open(results_md, "w") as f:
        f.write("\n".join(lines) + "\n")


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    vocab_sizes = preprocess(CFG)
    print(f"Vocab sizes (26 field): {vocab_sizes}")

    train_arr = load_arrays(CFG.OUTPUT_DIR, "train", CFG.SAMPLE_FRAC)
    test_arr = load_arrays(CFG.OUTPUT_DIR, "test", CFG.SAMPLE_FRAC)
    print(f"Train rows: {train_arr.n} | Test rows: {test_arr.n}")

    results_csv = os.path.join(CFG.OUTPUT_DIR, "results.csv")
    results_md = os.path.join(CFG.OUTPUT_DIR, "results.md")
    done = _done_models(results_csv) if CFG.RESUME else set()

    for name in CFG.MODELS:
        if name in done:
            print(f"[skip] {name} đã hoàn tất.")
            continue
        print(f"[train] {name} ...")
        res = train_model(name, train_arr, test_arr, vocab_sizes, CFG, device,
                          CFG.OUTPUT_DIR)
        print(f"[done] {name}: AUC={res['auc']:.5f} LogLoss={res['logloss']:.5f} "
              f"EmbParams={res['emb_params']:,} TotParams={res['n_params']:,} "
              f"FLOPs={res['flops']:,} RAM={res['ram_gb']:.2f}GB VRAM={res['vram_gb']:.2f}GB "
              f"h/epoch={res['h_per_epoch']:.4f} Infer={res['infer_ms']:.2f}ms/bs")
        _append_result(results_csv, res)
        _write_markdown(results_csv, results_md)

    print(f"Kết quả: {results_csv}\n{results_md}")


main()

Device: cuda
Vocab sizes (26 field): [1461, 578, 8378274, 1884136, 306, 25, 12495, 634, 4, 88846, 5655, 6950098, 3195, 28, 14756, 4589441, 11, 5592, 2169, 5, 5893095, 19, 16, 258696, 106, 133144]
Train rows: 36665662 | Test rows: 4591393
[train] deepfm ...
[done] deepfm: AUC=0.75158 LogLoss=0.56928 EmbParams=119,946,713 TotParams=120,439,929 FLOPs=984,026 RAM=22.01GB VRAM=4.24GB h/epoch=0.0381 Infer=1.37ms/bs
[train] dcn ...
[done] dcn: AUC=0.80338 LogLoss=0.44747 EmbParams=112,891,024 TotParams=113,226,828 FLOPs=667,432 RAM=22.08GB VRAM=3.19GB h/epoch=0.0279 Infer=0.80ms/bs
[train] dlrm ...
[done] dlrm: AUC=0.80384 LogLoss=0.44706 EmbParams=112,891,024 TotParams=113,056,801 FLOPs=329,984 RAM=22.19GB VRAM=3.19GB h/epoch=0.0283 Infer=0.76ms/bs
Kết quả: /kaggle/working/results.csv
/kaggle/working/results.md


## 9. Kết quả

In [15]:
print(open(os.path.join(CFG.OUTPUT_DIR, 'results.md')).read())

# Kết quả Bloom Embedding CTR

| Model | AUC | LogLoss | Emb Params | Tổng Params | FLOPs | RAM (GB) | VRAM (GB) | Train (h/ep) | Infer (ms/bs) | Epochs |
|---|---|---|---|---|---|---|---|---|---|---|
| deepfm | 0.75158 | 0.56928 | 0.1199B | 0.1204B | 984,026 | 22.01 | 4.24 | 0.0381 | 1.37 | 1 |
| dcn | 0.80338 | 0.44747 | 0.1129B | 0.1132B | 667,432 | 22.08 | 3.19 | 0.0279 | 0.80 | 1 |
| dlrm | 0.80384 | 0.44706 | 0.1129B | 0.1131B | 329,984 | 22.19 | 3.19 | 0.0283 | 0.76 | 1 |

